# CME538 - Introduction to Data Science

## Assignment 8 - Data Science Life Cycle

### Learning Objectives

After completing this assignment, you should be able to:

- Apply the data science life cycle to a real-world prediction problem.
- Clean and filter observational data using appropriate domain constraints.
- Use exploratory data analysis to identify patterns, anomalies, and potentially informative variables.
- Engineer numerical and categorical features for predictive modelling.
- Prevent data leakage by separating training, validation, and test data appropriately.
- Build reproducible preprocessing workflows for numerical and categorical features.
- Fit and compare baseline and linear regression models using scikit-learn.
- Evaluate regression models using root mean squared error (RMSE).
- Use validation data to compare models and make modelling decisions.
- Evaluate a selected model once on a held-out test set.
- Interpret model performance and identify opportunities for further improvement.

### Marking Breakdown

| Question | Marks |
|---|---:|
| Question 1a | 1 |
| Question 1b | 1 |
| Question 1c | 1 |
| Question 1d | 1 |
| Question 2a | 1 |
| Question 2b | 1 |
| Question 2c | 1 |
| Question 3a | 1 |
| Question 3b | 1 |
| Question 3c | 1 |
| Question 3d | 1 |
| Question 4a | 1 |
| Question 4b | 1 |
| Question 4c | 1 |
| Question 4d | 1 |
| Question 4e | 1 |
| Question 4f | 1 |
| Code quality | 3 |
| **Total** | **20** |

### Code Quality

Code quality will be assessed across the complete notebook.

| Level | Points | Description |
|---|---:|---|
| **Developing** | 1 | Code produces the required results but may be difficult to follow, unnecessarily repetitive, poorly organized, or include excessive output. |
| **Competent** | 2 | Code is organized and readable, uses appropriate Pandas and scikit-learn operations, and produces concise, relevant outputs. |
| **Strong** | 3 | Code is clear, concise, well organized, and reproducible; uses Pandas and scikit-learn effectively; avoids unnecessary operations and output; and executes successfully from beginning to end. |

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

# Configure notebook plots
%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook")

# Overview

Imagine you are working as a **data scientist for a transportation analytics team in New York City**.

The team has access to historical taxi trip data containing information about thousands of rides across the city. They want to use this data to improve their ability to answer an important question:

> **How long will a taxi trip take?**

Your task is to develop a **regression model** that predicts the duration of a taxi trip using information available about the ride.

But before building a model, you will need to understand and prepare the data. You will investigate unusual observations, explore relationships between variables, engineer numerical and categorical features, and compare several modelling approaches.

Throughout the assignment, you will work through the major stages of the **Data Science Life Cycle**:

**Explore → Prepare → Engineer Features → Model → Evaluate → Improve**

By the end of the assignment, you will have developed and evaluated a complete predictive modelling workflow for taxi trip duration.

---

# The Data

To investigate these questions, you will work with data from **New York City yellow taxi trips**.

The original trip records are published by the **NYC Taxi & Limousine Commission (TLC)**. The complete dataset is very large, so for this assignment you will work with a simple random sample stored in:

`taxi.csv`

Each row represents a single taxi trip and contains information about **when and where the trip occurred, the number of passengers, the distance travelled, the fare, the payment method, and the tip**.

The dataset contains the following variables:

| Variable | Description |
|---|---|
| `pickup_datetime` | Date and time when the taxi meter was engaged |
| `dropoff_datetime` | Date and time when the taxi meter was disengaged |
| `pickup_lon` | Longitude of the pickup location |
| `pickup_lat` | Latitude of the pickup location |
| `dropoff_lon` | Longitude of the drop-off location |
| `dropoff_lat` | Latitude of the drop-off location |
| `passengers` | Number of passengers in the vehicle |
| `distance` | Trip distance in miles |
| `payment_method` | Payment type: `1` = credit card, `2` = cash, `3` = no charge, `4` = dispute |
| `surcharge` | Improvement surcharge associated with the trip |
| `tax` | State and municipal taxes |
| `fare` | Time-and-distance fare calculated by the meter |
| `tip` | Recorded tip amount for credit-card transactions; cash tips are not recorded |

Some of these variables may be useful predictors directly, while others may need to be **cleaned, transformed, or combined into new features**.

Part of your job as the data scientist will be to determine which information is useful for each prediction problem.

----

# 1. Data Selection and Cleaning

Before building any predictive model, we need to make sure that the data represent the trips we actually want to study.

Taxi trip records may contain locations outside our area of interest or coordinates that are not reasonable for New York City. Including these observations could distort our exploratory analysis and affect the models we build later.

We will therefore begin by loading the data and restricting our analysis to trips whose **pickup and drop-off locations are both within a reasonable geographic boundary for New York City**.

For this assignment, we will use the following boundaries:

- Longitude: `-74.03` to `-73.75`
- Latitude: `40.60` to `40.88`

Both endpoints should be included.

---

## Question 1a — Load and Filter the Taxi Data

Import `taxi.csv` as a Pandas DataFrame and assign it to a variable named `all_taxi`.

Then create a DataFrame named `taxi` containing only trips where **both the pickup and drop-off locations** fall within the geographic boundaries above.

In other words, a trip should remain in `taxi` only when:

- `pickup_lon` is between `-74.03` and `-73.75`;
- `pickup_lat` is between `40.60` and `40.88`;
- `dropoff_lon` is between `-74.03` and `-73.75`; and
- `dropoff_lat` is between `40.60` and `40.88`.

Use the **training data provided here only**; do not remove observations based on other variables yet.

In [ ]:
# Question 1a

# Load the complete taxi sample
all_taxi = ...

# Keep only trips whose pickup and drop-off coordinates
# both fall within the specified NYC geographic boundaries
taxi = ...

taxi.head()

---
## Question 1b — Visualize Pickup Density

Before restricting the analysis further, let's examine where taxi pickups occur across the city.

Create an interactive map using **Folium** and `HeatMap` showing the density of taxi pickup locations in the filtered `taxi` DataFrame.

The heatmap should:

- be centred approximately on New York City;
- use `pickup_lat` and `pickup_lon`;
- use a heatmap radius of `10`; and
- make areas with a high concentration of pickups easy to identify.

Use the visualization to look for spatial patterns in taxi activity.

You should expect to see a strong concentration of pickups in Manhattan, along with additional high-density areas associated with major transportation hubs.

In [ ]:
import folium
from folium.plugins import HeatMap

# Question 1b

# Create a base map centred on New York City
pickup_map = ...

# Add taxi pickup locations as a density heatmap
...

pickup_map

---
## Question 1c - Calculate Trip Duration

To predict how long a taxi trip will take, we first need to calculate the observed duration of each trip.

Convert the pickup and drop-off timestamps in `taxi` to Pandas datetime values.

Then create a new column named `duration` containing the trip duration in **seconds**.

Store `duration` as an integer.

Use the filtered `taxi` DataFrame from Question 1a.

In [ ]:
# Question 1c

# Convert pickup and drop-off timestamps to datetime
taxi["pickup_datetime"] = ...

taxi["dropoff_datetime"] = ...

# Calculate trip duration in seconds
taxi["duration"] = ...

taxi.head()

---

## Question 1d — Remove Implausible Trips

Real-world transportation data can contain observations that are impossible or highly implausible.

Before modelling trip duration, we will remove trips that do not satisfy reasonable operating conditions.

First, create a new column named `speed` containing the trip's average speed in **miles per hour**:

$$
\text{speed}
=
\frac{\text{distance}}{\text{duration in hours}}
$$

Then create a DataFrame named `clean_taxi` containing only trips that satisfy all of the following:

- `passengers > 0`
- `distance > 0`
- `duration >= 60` seconds
- `duration <= 3600` seconds
- `speed <= 100` miles per hour

These rules remove trips with invalid passenger counts, zero-distance trips, extremely short or long recorded durations, and implausibly high average speeds.

In [ ]:
# Question 1d

# Calculate average trip speed in miles per hour
taxi["speed"] = ...

# Keep only trips that satisfy all cleaning rules
clean_taxi = ...

clean_taxi.head()

In [ ]:
# Verification - do not modify

print(
    "All passenger counts are positive:",
    (clean_taxi["passengers"] > 0).all()
)

print(
    "All trip distances are positive:",
    (clean_taxi["distance"] > 0).all()
)

print(
    "All durations are between 60 and 3600 seconds:",
    (
        (clean_taxi["duration"] >= 60)
        & (clean_taxi["duration"] <= 3600)
    ).all()
)

print(
    "All average speeds are at most 100 mph:",
    (clean_taxi["speed"] <= 100).all()
)

---

## Restricting the Analysis to Manhattan

Our geographic filter in Question 1a removed coordinates outside the broader New York City area, but our modelling problem will focus specifically on trips that **start and end in Manhattan**.

To identify these trips accurately, we will use the official New York borough boundaries through GeoPandas.

Rather than approximating Manhattan using a rectangular latitude/longitude box, we will perform a spatial check using the Manhattan polygon.

A trip will be retained only when:

- its pickup point lies within Manhattan, and
- its drop-off point lies within Manhattan.

The borough boundaries are loaded using the `geodatasets` package, which provides the `nybb` dataset used in the GeoPandas documentation.

In [ ]:
#pip install geodatasets
import geodatasets

# Load New York City borough boundaries
boroughs = gpd.read_file(
    geodatasets.get_path("nybb")
)

# Convert the borough boundaries to latitude/longitude coordinates
boroughs = boroughs.to_crs(epsg=4326)

boroughs[
    ["BoroName", "geometry"]
].head()

Let's plot these quickly.

In [ ]:
# Visualize New York City borough boundaries
ax = boroughs.plot(
    column="BoroName",
    categorical=True,
    figsize=(8, 8),
    legend=True,
    edgecolor="black",
    linewidth=0.6,
    alpha=0.5
)

ax.set_title("New York City Borough Boundaries")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()

Next, we will convert the pickup and drop-off coordinates into geographic points.

The coordinate columns are longitude/latitude values, so their coordinate reference system is:

`EPSG:4326`

We will then test whether each point lies within the Manhattan boundary.

In [ ]:
# Extract the Manhattan boundary
manhattan_boundary = ...

# Create geographic pickup points
pickup_points = ...

# Create geographic drop-off points
dropoff_points = ...

# Keep only trips whose pickup AND drop-off are within Manhattan
manhattan_taxi = ...

print(
    f"Clean NYC-area trips: {len(clean_taxi):,}"
)

print(
    f"Trips starting and ending in Manhattan: "
    f"{len(manhattan_taxi):,}"
)

In [ ]:
# Visualize the retained Manhattan pickup locations

manhattan_pickups = gpd.GeoDataFrame(
    manhattan_taxi.copy(),
    geometry=gpd.points_from_xy(
        manhattan_taxi["pickup_lon"],
        manhattan_taxi["pickup_lat"]
    ),
    crs="EPSG:4326"
)

ax = boroughs.plot(
    figsize=(8, 8),
    facecolor="none",
    edgecolor="black",
    linewidth=0.8
)

manhattan_pickups.plot(
    ax=ax,
    markersize=1,
    alpha=0.15
)

ax.set_xlim(-74.03, -73.90)
ax.set_ylim(40.68, 40.89)
ax.set_title("Pickup Locations for Retained Manhattan Trips")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()

We now have a cleaned dataset containing trips that:

1. have geographically reasonable New York City coordinates;
2. satisfy the trip-quality cleaning rules; and
3. both start and end within Manhattan.

The resulting `manhattan_taxi` DataFrame will be used for the remainder of the assignment.

We will save a checkpoint so that the cleaned data can be reloaded without repeating the spatial filtering if needed.

In [ ]:
# Save the cleaned Manhattan taxi data
manhattan_taxi.to_csv(
    "manhattan_taxi.csv",
    index=False
)

---

## Question 1e — Summarize the Cleaning Process *(Ungraded)*

Summarize how the dataset changed during the data-selection and cleaning process.

Your summary should report:

- the number of trips in the original dataset;
- the number and percentage of trips removed by the trip-quality cleaning rules; and
- the number of trips retained after requiring both the pickup and drop-off locations to be within Manhattan.

Generate the summary using Python rather than manually entering the numbers.

In [ ]:
# Question 1e

original_trips = ...
clean_trips = ...
manhattan_trips = ...

removed_trips = ...
removed_percent = ...

print(
    f"Of the original {original_trips:,} trips, "
    f"{removed_trips:,} trips ({removed_percent:.1f}%) "
    f"were removed through data cleaning. "
    f"Of the remaining {clean_trips:,} trips, "
    f"{manhattan_trips:,} trips that started and ended "
    f"in Manhattan were selected for further analysis."
)

---

# Predicting Trip Duration

You are now working with taxi trips that passed our quality checks and started and ended within Manhattan.

Your first modelling task is to predict **trip duration**.

Imagine that this model will eventually be used when a passenger requests a taxi ride. Given information about the trip, we would like to estimate how long the ride is likely to take.

Before fitting a regression model, however, we need to decide which observations are appropriate for model development.

---

# 2. Exploratory Data Analysis

January 2016 included several unusual events that may have affected normal taxi activity.

For example:

- New Year's Day occurred on January 1;
- Martin Luther King Jr. Day occurred on January 18; and
- a major winter storm affected New York City later in the month.

Our goal is to develop a model that represents **typical taxi operations**, rather than unusual conditions associated with holidays or severe weather.

In this section, you will use exploratory data analysis to investigate daily taxi activity and identify dates that may not be representative of normal conditions.

---

## Question 2a — Extract the Pickup Date

Create a new column named `date` in `manhattan_taxi` containing the calendar date on which each trip began.

The new column should contain the pickup **date only**, without the time of day.

In [ ]:
# Question 2a

# Extract the pickup date without the time
manhattan_taxi["date"] = ...

manhattan_taxi.head()

---

## Question 2b — Identify Unusual Days

Severe weather can substantially reduce taxi activity because fewer people travel and road conditions become more difficult.

Using `manhattan_taxi`, calculate the **number of taxi trips for each pickup date** and create an appropriate visualization of daily trip counts.

Your visualization should include:

- pickup date on the x-axis;
- number of trips on the y-axis;
- a clear title; and
- readable axis labels.

Use the visualization to identify the dates that appear to have been most strongly affected by the January 2016 blizzard.

In [ ]:
# Question 2b

# Count the number of trips for each pickup date
daily_trip_counts = ...

# Create a visualization of daily taxi activity
plt.figure(figsize=(12, 5))

...

plt.xlabel("Pickup Date")
plt.ylabel("Number of Trips")
plt.title("Daily Manhattan Taxi Trips — January 2016")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

**Q2b: Your answer:**

<!--
Which dates appear to have been most strongly affected by the January 2016 blizzard?

Use evidence from your visualization to support your answer.
-->

---

## Question 2c — Create the Modelling Dataset

Based on the calendar and the exploratory analysis above, we will exclude the following atypical days:

- January 1–3;
- January 18; and
- January 23–26.

These dates include holiday periods and days associated with the major winter storm.

The remaining dates will be treated as typical operating days for this assignment.

Create a DataFrame named `final_taxi` containing only trips whose pickup dates fall on the typical dates defined below.

In [ ]:
# Dates excluded from the modelling dataset
atypical_days = [
    1, 2, 3,
    18,
    23, 24, 25, 26
]

typical_days = [
    day
    for day in range(1, 32)
    if day not in atypical_days
]

print("Typical days:")
print(typical_days)

In [ ]:
# Question 2c

# Keep only trips occurring on typical operating days
final_taxi = ...

final_taxi.head()

In [ ]:
# Verification - do not modify

print(
    "Q2c Answer - Only typical days retained:",
    final_taxi["pickup_datetime"]
    .dt.day
    .isin(typical_days)
    .all()
)

print(
    "No atypical days remain:",
    ~final_taxi["pickup_datetime"]
    .dt.day
    .isin(atypical_days)
    .any()
)

print(
    "Number of modelling trips:",
    len(final_taxi)
)

---
### Optional Exploration

At this stage in a real data-science workflow, we would usually investigate the modelling data more thoroughly before selecting features.

You are welcome to perform additional exploratory analysis below. This work is optional and will not be graded.

In [ ]:
# Optional exploratory analysis

---

# 3. Feature Engineering

We are now ready to prepare predictors for the trip-duration model.

Our candidate predictors include:

- pickup location;
- drop-off location;
- trip distance;
- time of day;
- day of the week; and
- a simplified spatial region describing where the trip begins.

Before examining these relationships in detail, we will divide the modelling data into:

- a **training set** for exploratory analysis, feature engineering, and model fitting;
- a **validation set** for comparing candidate models; and
- a **test set** reserved for the final evaluation.

This separation is important because decisions made during exploratory analysis and feature engineering should not be influenced by the test set.

In [ ]:
from sklearn.model_selection import train_test_split

# Split into 70% training and 30% temporary holdout
train, holdout = train_test_split(
    final_taxi,
    test_size=0.30,
    random_state=0
)

# Divide the holdout equally into validation and test sets
val, test = train_test_split(
    holdout,
    test_size=0.50,
    random_state=0
)

# Create independent copies
train = train.copy()
val = val.copy()
test = test.copy()

print(f"Training observations: {len(train):,}")
print(f"Validation observations: {len(val):,}")
print(f"Test observations: {len(test):,}")

---

## Question 3a — Explore Trip Duration Across Dates

Before engineering additional predictors, examine whether the distribution of trip duration changes across the typical dates retained for modelling.

Using the **training data only**, create a boxplot showing:

- pickup date on the x-axis; and
- trip duration in seconds on the y-axis.

Use the plot to assess whether trip-duration distributions appear reasonably similar across the retained dates.

In [ ]:
# Question 3a

plt.figure(figsize=(12, 6))

# Compare trip-duration distributions across pickup dates
...

plt.xlabel("Pickup Date")
plt.ylabel("Trip Duration (seconds)")
plt.title("Trip Duration by Pickup Date")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

**Q3a: Your answer:**

<!--
Briefly describe any differences you observe in trip-duration
distributions across the training dates.
-->

---

## Question 3b — Engineer Time-Based Features

The pickup timestamp contains useful information that is not directly available to a regression model.

Create a function named `add_time_features()` that adds the following variables:

- `hour` — integer pickup hour from `0` to `23`;
- `day` — day of the week, where Monday = `0` and Sunday = `6`;
- `weekend` — `1` for Saturday or Sunday and `0` otherwise;
- `period` — a categorical representation of time of day:
  - `"Early Morning"` for 12:00 AM–5:59 AM,
  - `"Daytime"` for 6:00 AM–5:59 PM,
  - `"Night"` for 6:00 PM–11:59 PM.

The function should:

1. create a copy of the input DataFrame;
2. add the four new variables; and
3. return the modified copy.

Apply the function consistently to the training, validation, and test sets.

In [ ]:
# Question 3b

def add_time_features(data):
    """Return a copy of data with engineered time-based features."""
    
    prepared = data.copy()

    # Integer pickup hour: 0 through 23
    prepared["hour"] = ...

    # Day of week: Monday = 0, Sunday = 6
    prepared["day"] = ...

    # 1 for Saturday or Sunday, otherwise 0
    prepared["weekend"] = ...

    # Time period:
    # "Early Morning" = 12:00 AM–5:59 AM
    # "Daytime" = 6:00 AM–5:59 PM
    # "Night" = 6:00 PM–11:59 PM
    prepared["period"] = ...

    return prepared


# Apply the same feature-engineering function to each split
train = add_time_features(train)
val = add_time_features(val)
test = add_time_features(test)

train[
    [
        "pickup_datetime",
        "hour",
        "day",
        "weekend",
        "period"
    ]
].head()

In [ ]:
# Verification - do not modify

print(
    "Q3b Answer - Hour values are valid:",
    train["hour"].between(0, 23).all()
)

print(
    "Day values are valid:",
    train["day"].between(0, 6).all()
)

print(
    "Weekend is binary:",
    set(train["weekend"].unique()).issubset({0, 1})
)

print(
    "Expected periods created:",
    set(train["period"].dropna().unique())
    == {
        "Early Morning",
        "Daytime",
        "Night"
    }
)

---

## Question 3c — Explore Speed Across Time Periods

Traffic conditions change throughout the day, so the average speed of taxi trips may also depend on the pickup time.

Using the **training data only**, compare the distribution of `speed` for trips beginning during:

- Early Morning;
- Daytime; and
- Night.

Create an overlaid density visualization that makes the three distributions easy to compare.

Then briefly describe the relationship you observe between time of day and average taxi speed.

In [ ]:
# Question 3c

plt.figure(figsize=(10, 6))

# Compare the speed distributions across the three time periods
...

plt.xlabel("Average Speed (mph)")
plt.ylabel("Density")
plt.title("Distribution of Taxi Speed by Time of Day")

plt.tight_layout()
plt.show()

**Q3c: Your answer:**

<!--
Briefly describe the relationship you observe between
time of day and average taxi speed.
-->

---

## Engineering a Pickup-Region Feature

Location is another important factor in taxi travel time.

Rather than treating every pickup coordinate independently, we will create a simplified spatial feature that divides Manhattan pickup locations into three regions.

To do this, we will:

1. fit a one-component Principal Component Analysis (PCA) model using the training pickup coordinates;
2. project each pickup location onto that one-dimensional spatial axis;
3. divide the **training projections** into three approximately equal-frequency groups; and
4. apply those same PCA parameters and region boundaries to the validation and test sets.

The important methodological point is that the PCA model and region boundaries must be learned from the **training data only**.

This prevents information from the validation or test sets from influencing feature construction.

In [ ]:
from sklearn.decomposition import PCA

# Fit PCA using training pickup coordinates only
pickup_pca = PCA(
    n_components=1
)

pickup_pca.fit(
    train[
        [
            "pickup_lon",
            "pickup_lat"
        ]
    ]
)

# Project training pickup locations onto PC1
train_pc1 = (
    pickup_pca
    .transform(
        train[
            [
                "pickup_lon",
                "pickup_lat"
            ]
        ]
    )
    .ravel()
)

print(
    "Variance explained by PC1:",
    f"{pickup_pca.explained_variance_ratio_[0]:.1%}"
)

We now divide the training projections into three groups containing approximately equal numbers of training observations.

The resulting boundaries will then be reused for every new observation.

In [ ]:
# Learn region boundaries from training data only
_, region_bins = pd.qcut(
    train_pc1,
    q=3,
    retbins=True,
    duplicates="drop"
)

# Extend the outer boundaries so that future observations
# slightly outside the training range can still be categorized
region_bins[0] = -np.inf
region_bins[-1] = np.inf

print("Region boundaries:")
print(region_bins)

In [ ]:
def add_region(data, pca_model, bins):
    """Return a copy of data with a PCA-based pickup region."""
    
    prepared = data.copy()

    # Project pickup coordinates onto the PCA axis
    pc1 = ...

    # Assign observations using the boundaries
    # learned from the training data
    prepared["region"] = ...

    return prepared


# Apply the training-derived spatial transformation once
train = add_region(
    train,
    pickup_pca,
    region_bins
)

val = add_region(
    val,
    pickup_pca,
    region_bins
)

test = add_region(
    test,
    pickup_pca,
    region_bins
)

train[
    [
        "pickup_lon",
        "pickup_lat",
        "region"
    ]
].head()

In [ ]:
# Verification - do not modify

print(
    "Three training regions created:",
    train["region"].nunique() == 3
)

print(
    "No missing training regions:",
    train["region"].isna().sum() == 0
)

print(
    "No missing validation regions:",
    val["region"].isna().sum() == 0
)

print(
    "No missing test regions:",
    test["region"].isna().sum() == 0
)

In [ ]:
plt.figure(figsize=(7, 8))

sns.scatterplot(
    data=train,
    x="pickup_lon",
    y="pickup_lat",
    hue="region",
    s=8,
    alpha=0.4
)

plt.xlabel("Pickup Longitude")
plt.ylabel("Pickup Latitude")
plt.title("PCA-Based Pickup Regions")

plt.legend(
    title="Region",
    markerscale=2
)

plt.tight_layout()
plt.show()

---

## Question 3d — Explore Speed Across Pickup Regions

Location may interact with traffic conditions.

Using the **training data only**, focus on trips beginning during the **Early Morning** period and compare the distribution of `speed` across the three pickup regions.

Create an appropriate visualization and describe whether average trip speed appears to differ across regions.

In [ ]:
# Question 3d

# Restrict the comparison to early-morning trips
early_morning_train = ...

plt.figure(figsize=(10, 6))

# Compare speed distributions across pickup regions
...

plt.xlabel("Average Speed (mph)")
plt.ylabel("Density")
plt.title("Early-Morning Taxi Speed by Pickup Region")

plt.tight_layout()
plt.show()

**Q3d: Your answer:**

<!--
Do the three pickup regions appear to have different
early-morning speed distributions? Briefly describe what you observe.
-->

---

## Preparing Features for Modelling

Our regression model will use both numerical and categorical predictors.

### Numerical predictors

- `pickup_lon`
- `pickup_lat`
- `dropoff_lon`
- `dropoff_lat`
- `distance`

### Categorical predictors

- `hour`
- `day`
- `region`

We will use a scikit-learn `ColumnTransformer` so that:

- numerical predictors are standardized using `StandardScaler`; and
- categorical predictors are represented using `OneHotEncoder`.

Using a fitted preprocessing object is safer than independently calling `pd.get_dummies()` on the training, validation, and test sets because it guarantees that the same transformations and encoded columns are used for every dataset.

We will configure `OneHotEncoder` with `handle_unknown="ignore"` so that new observations can still be transformed if they contain a category that was absent when the encoder was fitted.

Two engineered variables will intentionally **not** be included:

- `period`, because it is derived directly from `hour`;
- `weekend`, because it is derived directly from `day`.

Most importantly, `speed` will **not** be used to predict `duration`.

Because

$$
\text{speed}
=
\frac{\text{distance}}{\text{duration}},
$$

including `speed` would provide the model with information derived from the target itself. This is a form of **target leakage**.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

# Numerical predictors
numeric_features = [
    "pickup_lon",
    "pickup_lat",
    "dropoff_lon",
    "dropoff_lat",
    "distance"
]

# Categorical predictors
categorical_features = [
    "hour",
    "day",
    "region"
]

# Create preprocessing steps for numerical and categorical features
trip_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            ...,
            numeric_features
        ),
        (
            "categorical",
            ...,
            categorical_features
        )
    ]
)

In [ ]:
# Prepare predictors and targets for modelling

X_train = ...
y_train = ...

X_val = ...
y_val = ...

X_test = ...
y_test = ...

print("Training predictors:", X_train.shape)
print("Validation predictors:", X_val.shape)
print("Test predictors:", X_test.shape)

In [ ]:
# Inspect the transformed training feature matrix

X_train_transformed = (
    trip_preprocessor
    .fit_transform(X_train)
)

print(
    "Original feature count:",
    X_train.shape[1]
)

print(
    "Transformed feature count:",
    X_train_transformed.shape[1]
)

print(
    "Training observations preserved:",
    X_train_transformed.shape[0]
    == X_train.shape[0]
)

---
# 4. Model Selection

We now have a cleaned modelling dataset and a reproducible preprocessing workflow.

Our next goal is to compare several regression approaches for predicting taxi trip duration.

We will evaluate candidate models using the **validation set**. The held-out test set will remain untouched until we have selected the final model.

We will compare:

1. a constant baseline model;
2. a simple linear regression using distance only;
3. a multiple linear regression using the full feature set;
4. a regularized Ridge regression model; and
5. a linear model that predicts the logarithm of trip duration.

All models will be compared using **Root Mean Squared Error (RMSE)**.

Lower RMSE indicates better predictive performance.

In [ ]:
def rmse(actual, predicted):
    """Calculate root mean squared error."""
    
    return np.sqrt(
        np.mean(
            (np.asarray(actual) - np.asarray(predicted)) ** 2
        )
    )

In [ ]:
# Question 4a

# Learn the constant prediction from the training target only
mean_training_duration = ...

# Predict the same value for every validation observation
constant_predictions = ...

# Calculate validation RMSE
constant_rmse = ...

print(
    f"Constant-model validation RMSE: "
    f"{constant_rmse:,.0f} seconds"
)

---

## Question 4b — Fit a Distance-Only Linear Model

Trip distance should contain substantial information about trip duration.

Fit a simple `LinearRegression` model using only `distance` as the predictor.

Use the training data to fit the model and the validation data to evaluate it.

Store the validation RMSE in:

`simple_rmse`

In [ ]:
from sklearn.linear_model import LinearRegression

# Question 4b

# Create and fit a distance-only linear regression model
simple_model = ...

simple_model.fit(
    ...,
    ...
)

# Predict trip duration for the validation set
simple_val_predictions = ...

# Calculate validation RMSE
simple_rmse = ...

print(
    f"Distance-only validation RMSE: "
    f"{simple_rmse:,.0f} seconds"
)

---

## Question 4c — Fit the Full Linear Regression Model

Distance alone does not capture differences in:

- pickup and drop-off location;
- time of day;
- day of the week; or
- pickup region.

Create a scikit-learn `Pipeline` named `linear_model` that combines:

1. the `trip_preprocessor` created in Section 3; and
2. `LinearRegression`.

Fit the pipeline using `X_train` and `y_train`.

Then predict the validation observations and store the validation RMSE in:

`linear_rmse`

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# Question 4c

# Combine preprocessing and linear regression in one pipeline
linear_model = Pipeline(
    steps=[
        (
            "preprocessor",
            ...
        ),
        (
            "regression",
            ...
        )
    ]
)

# Fit using the training data
linear_model.fit(
    ...,
    ...
)

# Predict the validation observations
linear_val_predictions = ...

# Calculate validation RMSE
linear_rmse = ...

print(
    f"Multiple linear regression validation RMSE: "
    f"{linear_rmse:,.0f} seconds"
)

---
## Question 4d — Add Regularization

Our full linear model contains many encoded predictors.

One way to reduce sensitivity to individual coefficients is to use **Ridge regression**, which adds L2 regularization.

The Ridge hyperparameter `alpha` controls the strength of regularization:

- smaller `alpha` → weaker regularization;
- larger `alpha` → stronger regularization.

Evaluate the following candidate values using the **validation set**:

```python
[0.1, 1, 10, 100]
```

For each value:
1. create a preprocessing-and-Ridge pipeline;
2. fit the model using the training data;
3. calculate validation RMSE.

Store the results in a DataFrame named ridge_results.

Then create:
1. best_alpha — the value with the lowest validation RMSE;
2. ridge_model — a pipeline using that value;
3. ridge_rmse — its validation RMSE.

In [ ]:
from sklearn.linear_model import Ridge

# Question 4d

ridge_alphas = [
    0.1,
    1,
    10,
    100
]

ridge_scores = []

# Evaluate each regularization strength
for alpha in ridge_alphas:

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                ...
            ),
            (
                "regression",
                ...
            )
        ]
    )

    # Fit using training data
    ...

    # Generate validation predictions
    predictions = ...

    # Store validation RMSE
    ridge_scores.append(
        ...
    )


ridge_results = pd.DataFrame({
    "alpha": ridge_alphas,
    "validation_rmse": ridge_scores
})

ridge_results

In [ ]:
# Select the alpha with the lowest validation RMSE

best_row = ...

best_alpha = ...
ridge_rmse = ...

# Create the final Ridge pipeline using the selected alpha
ridge_model = Pipeline(
    steps=[
        (
            "preprocessor",
            ...
        ),
        (
            "regression",
            ...
        )
    ]
)

ridge_model.fit(
    ...,
    ...
)

print(
    "Best alpha:",
    best_alpha
)

print(
    f"Best Ridge validation RMSE: "
    f"{ridge_rmse:,.0f} seconds"
)

---

## Question 4e — Model Log-Transformed Trip Duration

Trip duration is typically right-skewed: most trips are relatively short, while a smaller number take much longer.

A transformation of the target can sometimes make the relationship easier for a linear model to represent.

We will therefore compare a model that learns:

$$
\log(1 + \text{duration})
$$

instead of modelling duration directly.

Scikit-learn's `TransformedTargetRegressor` allows us to apply this transformation during fitting and automatically convert predictions back to seconds.

Create a model named `log_duration_model` that combines:

- `trip_preprocessor`;
- `LinearRegression`; and
- a `log1p` transformation of the target.

Calculate its validation RMSE and store it in:

`log_duration_rmse`

In [ ]:
from sklearn.compose import TransformedTargetRegressor

# Question 4e

# Build the preprocessing + linear regression pipeline
log_regressor = Pipeline(
    steps=[
        (
            "preprocessor",
            ...
        ),
        (
            "regression",
            ...
        )
    ]
)

# Apply log1p to the target during fitting and
# automatically transform predictions back to seconds
log_duration_model = TransformedTargetRegressor(
    regressor=...,
    func=...,
    inverse_func=...
)

# Fit using the training data
...

# Predict validation duration in seconds
log_val_predictions = ...

# Calculate validation RMSE
log_duration_rmse = ...

print(
    f"Log-duration validation RMSE: "
    f"{log_duration_rmse:,.0f} seconds"
)

The transformation can be understood by comparing the distributions of the original and transformed target.

The logarithm compresses the long right tail of the duration distribution, making unusually long trips less dominant during model fitting.

In [ ]:
fig, axes = plt.subplots(
    nrows=2,
    figsize=(10, 7)
)

sns.histplot(
    train["duration"],
    bins=40,
    kde=True,
    ax=axes[0]
)

axes[0].set_xlabel(
    "Trip Duration (seconds)"
)

axes[0].set_title(
    "Distribution of Trip Duration"
)

sns.histplot(
    np.log1p(
        train["duration"]
    ),
    bins=40,
    kde=True,
    ax=axes[1]
)

axes[1].set_xlabel(
    "log(1 + Trip Duration)"
)

axes[1].set_title(
    "Distribution of Log-Transformed Trip Duration"
)

plt.tight_layout()
plt.show()

---

## Question 4f — Select and Evaluate the Final Model

We have now compared several modelling approaches using the validation set.

Create a DataFrame named `model_comparison` containing the validation RMSE of:

- Constant Baseline
- Distance-Only Linear Regression
- Multiple Linear Regression
- Ridge Regression
- Log-Duration Linear Regression

Use the validation results to select the model with the lowest RMSE.

Store:

- the selected model in `final_model`;
- its name in `final_model_name`; and
- its validation RMSE in `final_validation_rmse`.

Only after selecting the model should you evaluate it on the held-out test set.

Fit the selected modelling approach using the combined **training and validation data**, evaluate it once on the test set, and store the final test RMSE in:

`test_rmse`

Do not make further modelling changes after examining the test-set result.

In [ ]:
# Question 4f

# Compare all candidate models using validation RMSE
model_comparison = pd.DataFrame({
    "Model": [
        "Constant Baseline",
        "Distance-Only Linear Regression",
        "Multiple Linear Regression",
        "Ridge Regression",
        "Log-Duration Linear Regression"
    ],
    "Validation RMSE": [
        ...,
        ...,
        ...,
        ...,
        ...
    ]
})

# Order models from lowest to highest validation RMSE
model_comparison = ...

model_comparison

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=model_comparison,
    x="Validation RMSE",
    y="Model"
)

plt.xlabel(
    "Validation RMSE (seconds)"
)

plt.ylabel("")

plt.title(
    "Validation Performance of Candidate Models"
)

plt.tight_layout()
plt.show()

In [ ]:
# Select the model with the lowest validation RMSE

final_model_name = ...
final_validation_rmse = ...

# Store the corresponding fitted modelling approach
if final_model_name == "Distance-Only Linear Regression":
    final_model = ...
elif final_model_name == "Multiple Linear Regression":
    final_model = ...
elif final_model_name == "Ridge Regression":
    final_model = ...
elif final_model_name == "Log-Duration Linear Regression":
    final_model = ...
else:
    final_model = None

print(
    "Selected model:",
    final_model_name
)

print(
    f"Selected validation RMSE: "
    f"{final_validation_rmse:,.0f} seconds"
)

In [ ]:
# Combine training and validation observations
# before fitting the final modelling approach

train_val = ...

X_train_val = ...

y_train_val = ...

In [ ]:
# Evaluate the selected modelling approach once on the test set

if final_model_name == "Constant Baseline":

    final_prediction = ...

    y_test_predicted = ...

elif final_model_name == "Distance-Only Linear Regression":

    final_model = ...

    final_model.fit(
        ...,
        ...
    )

    y_test_predicted = ...

else:

    # Refit the selected pipeline/model using all development data
    final_model.fit(
        ...,
        ...
    )

    y_test_predicted = ...


# Calculate the final held-out test RMSE
test_rmse = ...

print(
    f"Final validation RMSE: "
    f"{final_validation_rmse:,.0f} seconds"
)

print(
    f"Final test RMSE: "
    f"{test_rmse:,.0f} seconds"
)

In [ ]:
# Verification - do not modify

print(
    "Q4f Answer - Test predictions match test observations:",
    len(y_test_predicted) == len(y_test)
)

print(
    "Test RMSE is positive:",
    test_rmse > 0
)

print(
    "Test set evaluated only after model selection:",
    final_model_name
    in model_comparison["Model"].tolist()
)

### Final Reflection

**Your answer:**

<!--
Briefly compare the validation and test performance of your selected model.

Did the final test performance appear consistent with what you observed
during model development?

What does the final RMSE tell you about the model's prediction error?
-->

---

# Submission

Before submitting your assignment:

1. Restart the kernel and run all cells from beginning to end. Make sure the notebook executes without errors.

2. Submit your completed notebook (`.ipynb`) to **Quercus** and push the completed assignment to your **GitHub repository**.

3. Make sure your submitted notebook contains the required code, outputs, figures, and written responses, with unnecessary scratch cells removed.